# TP Jour 2 — Modèle d'Exécution Spark, Jointures Multi-Datasets & Couche Silver (Delta Lake)



In [8]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

print("Session Spark active — Spark UI : http://localhost:4040")

Session Spark active — Spark UI : http://localhost:4040


26/11/18 08:39:20 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 1. Chargement de vos tables Bronze (Multi-Datasets / Multi-Tables) & Diagnostic Qualité

Les trois tables Bronze ont été chargées au format Delta : `flights` contient **5 819 079 vols**, `airlines` **14 compagnies** et `airports` **322 aéroports**.

La table `flights` est utilisée comme source principale, tandis que les deux référentiels servent à ajouter les noms des compagnies et les informations des aéroports.


In [9]:
BRONZE_TABLES: dict[str, str] = {
    "main": "/lakehouse/bronze/flights",
    "airlines": "/lakehouse/bronze/airlines",
    "airports": "/lakehouse/bronze/airports",
}

assert len(BRONZE_TABLES) > 0, "TODO : Renseignez vos chemins Delta Bronze dans `BRONZE_TABLES`."
bronze_dfs = {alias: spark.read.format("delta").load(path) for alias, path in BRONZE_TABLES.items()}

for alias, df in bronze_dfs.items():
    print(f"[{alias}] {BRONZE_TABLES[alias]} -> {df.count():,} lignes | colonnes : {df.columns[:8]}")

df_bronze = bronze_dfs.get("main")





[main] /lakehouse/bronze/flights -> 5,819,079 lignes | colonnes : ['year', 'month', 'day', 'day_of_week', 'airline', 'flight_number', 'tail_number', 'origin_airport']
[airlines] /lakehouse/bronze/airlines -> 14 lignes | colonnes : ['iata_code', 'airline', 'source_file', '_ingested_at', '_source_dataset']
[airports] /lakehouse/bronze/airports -> 322 lignes | colonnes : ['iata_code', 'airport', 'city', 'state', 'country', 'latitude', 'longitude', 'source_file']


Avant de nettoyer les données Bronze, les valeurs NULL sont comptées dans chaque colonne avec `isNull()`, puis additionnées avec `sum()`. Ce contrôle permet de repérer les colonnes incomplètes et d'éviter de supprimer ou remplacer des valeurs sans connaître leur importance.

Le résultat montre notamment **105 071 retards à l'arrivée** et **86 153 retards au départ** manquants. Un second diagnostic est donc effectué pour vérifier si ces absences sont liées aux vols annulés ou déroutés, avant de choisir leur traitement.

In [10]:
# Compter les valeurs NULL dans chaque colonne de la table principale
null_report = df_bronze.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_bronze.columns
])

assert null_report is not None, "TODO : Calculez le diagnostic de valeurs nulles `null_report`."
null_report.show(truncate=False)


+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+-----------+------------+---------------+
|year|month|day|day_of_week|airline|flight_number|tail_number|origin_airport|destination_airport|scheduled_departure|departure_time|departure_delay|taxi_out|wheels_off|scheduled_time|elapsed_time|air_time|distance|wheels_on|taxi_in|scheduled_arrival|arrival_time|arrival_delay|diverted|cancelled|cancellation_reason|air_system_delay|security_delay|airline_delay|late_aircraft_delay|weather_delay|source_file|_ingested_at|_source_dataset|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+-------

Les vols sont regroupés par `cancelled` et `diverted` pour comprendre à quels statuts correspondent les retards manquants. Les **105 071 NULL à l'arrivée** concernent uniquement les **89 884 vols annulés** et les **15 187 vols déroutés**. Les **86 153 NULL au départ** concernent des vols annulés. Aucun retard n'est manquant parmi les vols ni annulés ni déroutés.

Les NULL sont conservés : les remplacer par zéro inventerait un retard nul, tandis que supprimer les lignes ferait perdre les vols concernés. Les analyses de retard devront donc distinguer les retards renseignés des retards manquants.

In [11]:
# Analyse complémentaire : exploiter les NULL observés dans le diagnostic

null_context = (
    df_bronze
    .groupBy("cancelled", "diverted")
    .agg(
        F.count("*").alias("vols"),
        F.sum(
            F.col("arrival_delay").isNull().cast("long")
        ).alias("arrival_delay_null"),
        F.sum(
            F.col("departure_delay").isNull().cast("long")
        ).alias("departure_delay_null")
    )
    .orderBy("cancelled", "diverted")
)

null_context.show(truncate=False)


+---------+--------+-------+------------------+--------------------+
|cancelled|diverted|vols   |arrival_delay_null|departure_delay_null|
+---------+--------+-------+------------------+--------------------+
|0        |0       |5714008|0                 |0                   |
|0        |1       |15187  |15187             |0                   |
|1        |0       |89884  |89884             |86153               |
+---------+--------+-------+------------------+--------------------+



Avant les jointures, les codes IATA des référentiels sont normalisés avec trim() et upper(), puis contrôlés pour détecter d'éventuels doublons. Cette vérification permet de repérer les clés qui pourraient multiplier les lignes de vols lors de l'enrichissement. Les référentiels sont ensuite dédoublonnés sur ces clés.

In [12]:
# Vérifier les doublons après normalisation des codes IATA

for alias in ["airlines", "airports"]:
    df_normalized = bronze_dfs[alias].withColumn(
        "iata_code",
        F.upper(F.trim(F.col("iata_code")))
    )

    duplicates = (
        df_normalized
        .groupBy("iata_code")
        .count()
        .filter(F.col("count") > 1)
    )

    print(f"Doublons après nettoyage dans {alias} :")
    duplicates.show(truncate=False)

Doublons après nettoyage dans airlines :


+---------+-----+
|iata_code|count|
+---------+-----+
+---------+-----+

Doublons après nettoyage dans airports :
+---------+-----+
|iata_code|count|
+---------+-----+
+---------+-----+



## 2. Harmonisation Inter-Datasets, Nettoyage & Jointure Distribuée (`BroadcastHashJoin` / `SortMergeJoin`)


Les codes des compagnies et des aéroports sont harmonisés avec `trim()` et `upper()` pour éviter que des espaces ou des différences de casse empêchent les correspondances. Le nombre de vols est ensuite relevé avant les jointures pour vérifier qu'aucune ligne n'est perdue ou dupliquée.

In [13]:
SILVER_MAIN_PATH = "/lakehouse/silver/flights_enriched"  # TODO : Définir le chemin cible de votre table Silver enrichie

# 1. Nettoyer et harmoniser les codes de la table principale
# On évite dropDuplicates() sur toutes les colonnes : opération coûteuse en mémoire.
df_flights = bronze_dfs["main"]

for col_name in ["airline", "origin_airport", "destination_airport"]:
    df_flights = df_flights.withColumn(
        col_name,
        F.upper(F.trim(F.col(col_name)))
    )

# Vérifier le nombre de lignes avant les jointures
count_before_join = df_flights.count()
print(f"Nombre de vols avant jointures : {count_before_join:,}")

Nombre de vols avant jointures : 5,819,079


Les codes des référentiels sont normalisés de la même manière que ceux des vols, puis dédoublonnés pour éviter plusieurs correspondances sur une même clé. Le référentiel des aéroports est préparé sous deux formes, une pour le départ et une pour l'arrivée, avec des noms de colonnes distincts.

In [14]:
# 2. Préparer les référentiels (petites tables)

# Compagnies aériennes
df_airlines = (
    bronze_dfs["airlines"]
    .withColumn("iata_code", F.upper(F.trim(F.col("iata_code"))))
    .dropDuplicates(["iata_code"])
    .select(
        F.col("iata_code").alias("airline_code"),
        F.col("airline").alias("airline_name")
    )
)

# Aéroports
df_airports = (
    bronze_dfs["airports"]
    .withColumn("iata_code", F.upper(F.trim(F.col("iata_code"))))
    .dropDuplicates(["iata_code"])
)

# Aéroports de départ
df_origin = df_airports.select(
    F.col("iata_code").alias("origin_code"),
    F.col("airport").alias("origin_airport_name"),
    F.col("city").alias("origin_city"),
    F.col("state").alias("origin_state")
)

# Aéroports d'arrivée
df_destination = df_airports.select(
    F.col("iata_code").alias("destination_code"),
    F.col("airport").alias("destination_airport_name"),
    F.col("city").alias("destination_city"),
    F.col("state").alias("destination_state")
)

Les trois jointures ajoutent les informations des compagnies et des aéroports aux vols. Le type `left` permet de conserver les vols même lorsqu'aucune correspondance n'est trouvée. Comme les référentiels sont petits, `broadcast()` évite de redistribuer la grande table de vols entre les tâches Spark.

In [15]:
# 3. Effectuer les jointures avec broadcast
# Les référentiels sont petits : Spark peut les diffuser sans redistribuer les vols.
df_silver = (
    df_flights
    .join(
        F.broadcast(df_airlines),
        F.col("airline") == F.col("airline_code"),
        "left"
    )
    .join(
        F.broadcast(df_origin),
        F.col("origin_airport") == F.col("origin_code"),
        "left"
    )
    .join(
        F.broadcast(df_destination),
        F.col("destination_airport") == F.col("destination_code"),
        "left"
    )
    .drop("airline_code", "origin_code", "destination_code")
)

Trois colonnes métier sont ajoutées pour faciliter les analyses : `is_delayed_15min` pour identifier les vols ayant au moins 15 minutes de retard à l'arrivée, `arrival_delay_minutes` pour mesurer les minutes de retard effectives sans comptabiliser les arrivées en avance, et `flight_date` pour analyser les retards et les annulations par période. Les valeurs NULL sont conservées dans les indicateurs de retard afin de ne pas confondre une information manquante avec une absence de retard.

In [16]:
# 4. Créer les colonnes métier dérivées

df_silver = (
    df_silver

    # Identifier les vols ayant au moins 15 minutes de retard
    .withColumn(
        "is_delayed_15min",
        F.when(F.col("arrival_delay").isNull(), F.lit(None).cast("int"))
         .when(F.col("arrival_delay") >= 15, 1)
         .otherwise(0)
    )

    # Mesurer uniquement les minutes de retard positif
    .withColumn(
        "arrival_delay_minutes",
        F.when(F.col("arrival_delay").isNull(), F.lit(None))
         .otherwise(F.greatest(F.col("arrival_delay"), F.lit(0)))
    )

    # Regrouper les composantes de date
    .withColumn(
        "flight_date",
        F.make_date(
            F.col("year").cast("int"),
            F.col("month").cast("int"),
            F.col("day").cast("int")
        )
    )
)

Le plan d'exécution est affiché avec `explain()` pour vérifier la stratégie de jointure utilisée par Spark. La table enrichie est ensuite enregistrée au format Delta. Le mode `overwrite` remplace son contenu et `overwriteSchema` permet de remplacer également son schéma.

In [17]:
# 5. Vérifier les correspondances et les éventuelles duplications
# Les contrôles sont effectués après l'écriture pour éviter des calculs répétés.

assert df_silver is not None, "TODO : Construisez le DataFrame nettoyé et joint `df_silver`."
df_silver.explain("simple")


# TODO : Sauvegarder `df_silver` au format "delta" (mode overwrite + overwriteSchema=true) dans `SILVER_MAIN_PATH`
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_MAIN_PATH)
)


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [year#6345, month#6346, day#6347, day_of_week#6348, airline#9453, flight_number#6350, tail_number#6351, origin_airport#9454, destination_airport#9455, scheduled_departure#6354, departure_time#6355, departure_delay#6356, taxi_out#6357, wheels_off#6358, scheduled_time#6359, elapsed_time#6360, air_time#6361, distance#6362, wheels_on#6363, taxi_in#6364, scheduled_arrival#6365, arrival_time#6366, arrival_delay#6367, diverted#6368, cancelled#6369, ... 19 more fields]
   +- BroadcastHashJoin [destination_airport#9455], [destination_code#9532], LeftOuter, BuildRight, false, false
      :- Project [year#6345, month#6346, day#6347, day_of_week#6348, airline#9453, flight_number#6350, tail_number#6351, origin_airport#9454, destination_airport#9455, scheduled_departure#6354, departure_time#6355, departure_delay#6356, taxi_out#6357, wheels_off#6358, scheduled_time#6359, elapsed_time#6360, air_time#6361, distance#6362, wheels_on#6363,

26/11/18 08:39:35 WARN MemoryManager: Total allocation exceeds 95.00% (2,040,109,440 bytes) of heap memory
Scaling row group sizes to 95.00% for 16 writers


La table Silver est relue pour vérifier le résultat réellement enregistré. Son nombre de lignes est comparé à celui des vols avant les jointures. Les deux comptages donnent **5 819 079 lignes**, ce qui confirme qu'aucun vol n'a été supprimé ou multiplié.

In [18]:
silver_saved = spark.read.format("delta").load(SILVER_MAIN_PATH)

silver_count = silver_saved.count()

# Vérifier l'absence d'explosion N-M
assert silver_count == count_before_join, (
    f"Alerte : {silver_count:,} lignes après jointure "
    f"contre {count_before_join:,} avant jointure !"
)

print(f"Table Silver écrite dans {SILVER_MAIN_PATH} : {silver_count:,} lignes")


Table Silver écrite dans /lakehouse/silver/flights_enriched : 5,819,079 lignes


Le taux de correspondance est calculé en comptant la proportion de noms retrouvés après les jointures. Toutes les compagnies sont identifiées, mais seulement **91,65 % des aéroports** le sont au départ comme à l'arrivée. Les codes des aéroports non retrouvés sont donc examinés dans le diagnostic suivant pour comprendre ces écarts.

In [19]:

# Vérifier le taux de rapprochement avec les référentiels
silver_saved.select(
    F.count("*").alias("total_flights"),
    F.round(
        F.avg(F.col("airline_name").isNotNull().cast("double")) * 100, 2
    ).alias("airlines_match_pct"),
    F.round(
        F.avg(F.col("origin_airport_name").isNotNull().cast("double")) * 100, 2
    ).alias("origins_match_pct"),
    F.round(
        F.avg(F.col("destination_airport_name").isNotNull().cast("double")) * 100, 2
    ).alias("destinations_match_pct")
).show(truncate=False)


+-------------+------------------+-----------------+----------------------+
|total_flights|airlines_match_pct|origins_match_pct|destinations_match_pct|
+-------------+------------------+-----------------+----------------------+
|5819079      |100.0             |91.65            |91.65                 |
+-------------+------------------+-----------------+----------------------+



In [20]:
# Diagnostic des aéroports sans correspondance

print("Aéroports de départ non retrouvés :")

(
    silver_saved
    .filter(F.col("origin_airport_name").isNull())
    .groupBy("origin_airport")
    .count()
    .orderBy(F.desc("count"))
    .show(20, truncate=False)
)

print("Aéroports d'arrivée non retrouvés :")

(
    silver_saved
    .filter(F.col("destination_airport_name").isNull())
    .groupBy("destination_airport")
    .count()
    .orderBy(F.desc("count"))
    .show(20, truncate=False)
)

Aéroports de départ non retrouvés :
+--------------+-----+
|origin_airport|count|
+--------------+-----+
|10397         |32588|
|13930         |27652|
|11298         |21044|
|11292         |18136|
|12892         |17728|
|14771         |14170|
|12266         |13211|
|14107         |12921|
|12889         |12688|
|13487         |10584|
|14747         |10388|
|10721         |10164|
|11433         |9925 |
|11618         |9622 |
|11057         |9572 |
|13204         |9047 |
|14869         |8763 |
|12953         |8590 |
|12478         |8303 |
|10821         |8022 |
+--------------+-----+
only showing top 20 rows
Aéroports d'arrivée non retrouvés :
+-------------------+-----+
|destination_airport|count|
+-------------------+-----+
|10397              |32594|
|13930              |27608|
|11298              |21033|
|11292              |18125|
|12892              |17739|
|14771              |14170|
|12266              |13227|
|14107              |12932|
|12889              |12702|
|13487         

In [23]:
# Vérifier les formats des codes d'aéroport sans correspondance.
silver_saved.filter(
    F.col("origin_airport_name").isNull()
).groupBy(
    F.when(
        F.col("origin_airport").rlike("^[0-9]+$"),
        "Identifiant numérique"
    ).otherwise("Autre format").alias("type_code")
).count().show()

+--------------------+------+
|           type_code| count|
+--------------------+------+
|Identifiant numér...|486165|
+--------------------+------+



Les jointures permettent de retrouver 100 % des compagnies aériennes et 91,65 % des aéroports de départ et d'arrivée.

Le diagnostic montre que les 486 165 vols sans correspondance pour l'aéroport de départ utilisent des identifiants numériques, alors que le référentiel `airports` est basé sur des codes IATA. Ces identifiants ne peuvent donc pas être associés directement aux noms des aéroports.

Les vols concernés sont conservés grâce aux jointures `left`. Pour compléter l'enrichissement, une table de correspondance entre les identifiants numériques et les codes IATA serait nécessaire.

## 3. Garanties Transactionnelles ACID de Delta Lake



Le fichier JSON de la version initiale est consulté dans `_delta_log` pour examiner les informations enregistrées lors de la création de Silver. Il permet notamment de retrouver l'opération d'écriture, le schéma de la table et les fichiers ajoutés.

Une écriture contenant deux colonnes étrangères au schéma de Silver est volontairement tentée pour vérifier son comportement. Delta Lake refuse l'ajout avec l'erreur `DELTA_METADATA_MISMATCH`, ce qui confirme que cette écriture incompatible est bloquée.

In [21]:

# 1. Lecture directe du journal transactionnel JSON sur HDFS
spark.read.text(
    f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json"
).show(8, truncate=110)

# 2. Tester le Schema Enforcement avec un DataFrame au mauvais schéma
bad_df = spark.createDataFrame(
    [("test_1", "colonne_intruse")],
    ["fake_id", "intruder_col"]
)

try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)
    raise AssertionError("Le schéma invalide a été accepté !")

except AssertionError:
    raise

except Exception as e:
    print(
        "Schema Enforcement validé — écriture invalide bloquée :",
        str(e).splitlines()[0]
    )


+--------------------------------------------------------------------------------------------------------------+
|                                                                                                         value|
+--------------------------------------------------------------------------------------------------------------+
|{"commitInfo":{"timestamp":1794908225020,"operation":"WRITE","operationParameters":{"mode":"Overwrite","par...|
|{"metaData":{"id":"28498c49-2129-48c8-aec8-8b3eea1264d0","format":{"provider":"parquet","options":{}},"sche...|
|                                                      {"protocol":{"minReaderVersion":1,"minWriterVersion":2}}|
|{"add":{"path":"part-00000-605ab6b8-8944-4905-a622-d859383bfbeb-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00001-f4a65030-1af7-4898-87cd-087aa5b31708-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00002-42af47c7-6c69-4206-96e0-ca1616777b8b-c000.snappy.parquet","partition

Le fichier JSON de la version initiale est consulté dans `_delta_log` pour examiner les informations enregistrées lors de l'écriture de Silver, notamment le schéma et les fichiers ajoutés. Une écriture avec deux colonnes incompatibles est ensuite tentée pour tester le Schema Enforcement. Delta Lake la refuse avec l'erreur `DELTA_METADATA_MISMATCH`, ce qui confirme que cette écriture ne peut pas modifier le schéma de la table.

### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)


In [22]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from functools import reduce
from operator import and_

MERGE_TEST_PATH = "/lakehouse/silver/flights_merge_test"

# 1. Définir les colonnes permettant d'identifier un vol
flight_keys = [
    "year",
    "month",
    "day",
    "airline",
    "flight_number",
    "origin_airport",
    "destination_airport",
    "scheduled_departure"
]

df_current = spark.read.format("delta").load(SILVER_MAIN_PATH)

# 2. Sélectionner une combinaison de clés unique
unique_flight = (
    df_current
    .groupBy(*flight_keys)
    .count()
    .filter(F.col("count") == 1)
    .select(*flight_keys)
    .limit(1)
    .collect()
)

assert unique_flight, "Aucun vol avec une clé unique trouvé."

selected_keys = spark.createDataFrame(
    unique_flight,
    df_current.select(*flight_keys).schema
)

# Utiliser une comparaison compatible avec les valeurs NULL
conditions = [
    df_current[col].eqNullSafe(selected_keys[col])
    for col in flight_keys
]

# Correction : left_semi évite de dupliquer les colonnes
df_test = (
    df_current
    .join(
        selected_keys,
        reduce(and_, conditions),
        "left_semi"
    )
    .withColumn("test_status", F.lit("before_merge"))
)

assert df_test.count() == 1

# 3. Créer une table Delta de test
(
    df_test.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(MERGE_TEST_PATH)
)

target_delta = DeltaTable.forPath(spark, MERGE_TEST_PATH)

# Version enregistrée avant le MERGE
version_before = target_delta.history(1).first()["version"]

# 4. Préparer une mise à jour réellement vérifiable
df_update = df_test.withColumn(
    "test_status",
    F.lit("after_merge")
)

merge_condition = " AND ".join(
    f"target.`{col}` <=> source.`{col}`"
    for col in flight_keys
)

# 5. Exécuter le MERGE
(
    target_delta.alias("target")
    .merge(
        df_update.alias("source"),
        merge_condition
    )
    .whenMatchedUpdate(
        set={
            "test_status": "source.test_status"
        }
    )
    .execute()
)

print("MERGE INTO exécuté.")

# 6. Consulter l'historique Delta
history_df = target_delta.history()

history_df.select(
    "version",
    "timestamp",
    "operation",
    "operationMetrics"
).show(truncate=False)

version_after = history_df.first()["version"]

# 7. Lire les versions avant et après
df_before = (
    spark.read
    .format("delta")
    .option("versionAsOf", version_before)
    .load(MERGE_TEST_PATH)
)

df_after = (
    spark.read
    .format("delta")
    .option("versionAsOf", version_after)
    .load(MERGE_TEST_PATH)
)

print("Avant MERGE :")
df_before.select(
    *flight_keys,
    "arrival_delay",
    "is_delayed_15min",
    "test_status"
).show(truncate=False)

print("Après MERGE :")
df_after.select(
    *flight_keys,
    "arrival_delay",
    "is_delayed_15min",
    "test_status"
).show(truncate=False)

# 8. Vérifier le changement effectif
assert df_before.first()["test_status"] == "before_merge"
assert df_after.first()["test_status"] == "after_merge"

print("MERGE et Time Travel validés.")

26/11/18 08:41:04 WARN MapPartitionsRDD: RDD 557 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


MERGE INTO exécuté.
+-------+-----------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                                                                                                                                                                                  

+----+-----+---+-------+-------------+--------------+-------------------+-------------------+-------------+----------------+------------+
|year|month|day|airline|flight_number|origin_airport|destination_airport|scheduled_departure|arrival_delay|is_delayed_15min|test_status |
+----+-----+---+-------+-------------+--------------+-------------------+-------------------+-------------+----------------+------------+
|2015|10   |25 |UA     |1122         |12266         |12339              |1309               |-7           |0               |before_merge|
+----+-----+---+-------+-------------+--------------+-------------------+-------------------+-------------+----------------+------------+

Après MERGE :
+----+-----+---+-------+-------------+--------------+-------------------+-------------------+-------------+----------------+-----------+
|year|month|day|airline|flight_number|origin_airport|destination_airport|scheduled_departure|arrival_delay|is_delayed_15min|test_status|
+----+-----+---+-----

Pour tester le `MERGE INTO` sans modifier les indicateurs métier de la table Silver, un vol identifié par une combinaison unique de huit colonnes est sélectionné et copié dans une table Delta de test.

Une colonne `test_status`, initialisée à `before_merge`, est ajoutée. Le MERGE permet ensuite de mettre à jour cette valeur vers `after_merge`.

L'historique Delta est consulté afin de récupérer les versions précédant et suivant le MERGE. Le Time Travel permet de comparer le même vol dans ces deux versions et de vérifier que la modification a effectivement été appliquée.

Cette méthode démontre le fonctionnement du MERGE et du versionnement Delta sans altérer les données métier de la table Silver principale.